In [3]:
!pip install geopy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.4/125.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.7/40.7 kB 3.9 MB/s eta 0:00:00


In [4]:
import csv
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderServiceError
import time

In [5]:
def get_location_info(city_name, geolocator, max_retries=3):
    """
    Get latitude, longitude, and state for a given city name.
    
    Args:
        city_name: Name of the city
        geolocator: Nominatim geolocator instance
        max_retries: Maximum number of retry attempts
    
    Returns:
        Dictionary with latitude, longitude, and state or None values
    """
    for attempt in range(max_retries):
        try:
            # Add a delay to respect rate limits
            time.sleep(1)
            
            location = geolocator.geocode(city_name, timeout=10, addressdetails=True)
            
            if not location:
                # Try with "USA" appended for US cities
                location = geolocator.geocode(f"{city_name}, USA", timeout=10, addressdetails=True)
            
            if location:
                address = location.raw.get('address', {})
                
                # Try to extract state from various address fields
                state = (address.get('state') or 
                        address.get('state_code') or 
                        address.get('region') or 
                        None)
                
                return {
                    'latitude': location.latitude,
                    'longitude': location.longitude,
                    'state': state
                }
            else:
                return {'latitude': None, 'longitude': None, 'state': None}
                
        except (GeocoderTimedOut, GeocoderServiceError) as e:
            if attempt < max_retries - 1:
                print(f"  Retry {attempt + 1} for {city_name}...")
                time.sleep(2)
            else:
                print(f"  Failed to geocode {city_name}: {e}")
                return {'latitude': None, 'longitude': None, 'state': None}
    
    return {'latitude': None, 'longitude': None, 'state': None}

def add_coordinates_to_csv(input_file, output_file='city_statistics_with_coords.csv'):
    """
    Read a CSV file with city names and counts, add geographical coordinates and state.
    
    Args:
        input_file: Path to the input CSV file
        output_file: Path to the output CSV file
    """
    # Initialize geolocator
    geolocator = Nominatim(user_agent="city_coordinates_app")
    
    # Read the input CSV
    cities_data = []
    with open(input_file, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            cities_data.append(row)
    
    print(f"Processing {len(cities_data)} cities...\n")
    
    # Add coordinates and state to each city
    results = []
    for i, row in enumerate(cities_data, 1):
        city = row['city']
        count = row['count']
        
        print(f"[{i}/{len(cities_data)}] Geocoding: {city}")
        
        location_info = get_location_info(city, geolocator)
        
        results.append({
            'city': city,
            'count': count,
            'state': location_info['state'] if location_info['state'] else '',
            'latitude': location_info['latitude'] if location_info['latitude'] is not None else '',
            'longitude': location_info['longitude'] if location_info['longitude'] is not None else ''
        })
        
        if location_info['latitude'] and location_info['longitude']:
            state_info = f", {location_info['state']}" if location_info['state'] else ""
            print(f"  ✓ Found: ({location_info['latitude']:.4f}, {location_info['longitude']:.4f}){state_info}")
        else:
            print(f"  ✗ Not found")
        print()
    
    # Write to output CSV
    with open(output_file, 'w', newline='', encoding='utf-8') as f:
        fieldnames = ['city', 'count', 'state', 'latitude', 'longitude']
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(results)
    
    print(f"\n{'='*60}")
    print(f"Results saved to: {output_file}")
    
    # Summary
    found = sum(1 for r in results if r['latitude'])
    not_found = len(results) - found
    with_state = sum(1 for r in results if r['state'])
    print(f"Successfully geocoded: {found}/{len(results)}")
    print(f"With state information: {with_state}/{len(results)}")
    print(f"Not found: {not_found}")
    print(f"{'='*60}")

In [12]:
def main():
    input_file = 'city_statistics.csv'
    output_file = 'city_statistics_with_coords.csv'
    
    print("City Coordinates Enrichment Tool")
    print("="*60)
    print()
    
    try:
        add_coordinates_to_csv(input_file, output_file)
    except FileNotFoundError:
        print(f"Error: Could not find '{input_file}'")
        print("Please make sure the file exists in the current directory.")
    except Exception as e:
        print(f"Error: {e}")

In [13]:
if __name__ == '__main__':
    main()

City Coordinates Enrichment Tool

Processing 66 cities...

[1/66] Geocoding: Washington
  ✓ Found: (38.8950, -77.0365), District of Columbia

[2/66] Geocoding: New York
  ✓ Found: (40.7127, -74.0060), New York

[3/66] Geocoding: Honolulu
  ✓ Found: (21.3045, -157.8557), Hawaii

[4/66] Geocoding: Chicago
  ✓ Found: (41.8756, -87.6244), Illinois

[5/66] Geocoding: Springfield
  ✓ Found: (39.7990, -89.6440), Illinois

[6/66] Geocoding: Richmond
  ✓ Found: (37.5385, -77.4343), Virginia

[7/66] Geocoding: Columbia
  ✓ Found: (4.0999, -72.9088)

[8/66] Geocoding: San Francisco
  ✓ Found: (37.7879, -122.4075), California

[9/66] Geocoding: Augusta
  ✓ Found: (48.3690, 10.8980), Bayern

[10/66] Geocoding: Omaha
  ✓ Found: (41.2587, -95.9384), Nebraska

[11/66] Geocoding: New Haven
  ✓ Found: (41.3082, -72.9251), Connecticut

[12/66] Geocoding: Milwaukee
  ✓ Found: (43.0386, -87.9091), Wisconsin

[13/66] Geocoding: Salt Lake City
  ✓ Found: (40.7596, -111.8868), Utah

[14/66] Geocoding: Wilming